## Interactive Alpha Earth Change Detection

In [1]:
# Setup: imports and Earth Engine init
import ee, geemap
from ipyleaflet import DrawControl, Marker, LayersControl
import ipywidgets as widgets
from ipywidgets import HTML, VBox, HBox, Label
import math
from datetime import datetime

print("Authenticating/initializing Earth Engine…")
try:
    ee.Initialize()
except Exception:
    print("No active EE session found. Launching authentication flow…")
    ee.Authenticate()
    ee.Initialize()

# Basic instructions
print("Draw a polygon/rectangle, set params, then click 'Compute change'.")

Authenticating/initializing Earth Engine…
Draw a polygon/rectangle, set params, then click 'Compute change'.


In [8]:
# Visualization parameters (user-specified)
# Year (2017–2024)
vis_yod = {'min': 2017, 'max': 2024,
           'palette': ['#2166ac','#4393c3','#92c5de','#fddbc7','#d6604d','#b2182b','#67001f']}
# Magnitude
vis_mag_a = {'min': 0, 'max': 0.5, 'palette': ['white','#fee08b','#f46d43','#a50026']}
vis_mag_l = {'min': 0, 'max': 800, 'palette': ['white','#fee08b','#f46d43','#a50026']}
# Duration
vis_dur_a = {'min': 0, 'max': 7, 'palette': ['#ffffff','#f3e5f5','#e1bee7','#ce93d8','#ba68c8','#9c27b0','#6a1b9a','#4a148c']}
vis_dur_l = {'min': 0, 'max': 7, 'palette': ['#ffffff','#f3e5f5','#e1bee7','#ce93d8','#ba68c8','#9c27b0','#6a1b9a','#4a148c']}

In [3]:
# Helpers: AlphaEarth change layers for a given AOI geometry
embeddings = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
years = list(range(2017, 2025))

def _normalize(img):
    bands = img.bandNames()
    norm = img.select(bands).pow(2).reduce(ee.Reducer.sum()).sqrt().add(1e-6)
    return img.select(bands).divide(norm)

def compute_alpha_layers(geom, change_threshold=0.15):
    # Load yearly images for AOI
    imgs = []
    for y in years:
        img_y = (embeddings
                 .filterDate(ee.Date.fromYMD(y, 1, 1), ee.Date.fromYMD(y+1, 1, 1))
                 .filterBounds(geom)
                 .mosaic()
                 .clip(geom))
        imgs.append(img_y)
    # Normalize all
    imgs_norm = [_normalize(i) for i in imgs]
    # Cosine similarity for consecutive pairs and carry end-year
    cos_list = []
    for i in range(len(imgs_norm) - 1):
        imgA = ee.Image(imgs_norm[i])
        imgB = ee.Image(imgs_norm[i+1])
        cos = imgA.multiply(imgB).reduce(ee.Reducer.sum()).rename('cos')
        yr = years[i+1]
        cos_with_year = cos.addBands(ee.Image.constant(yr).rename('pair_year').toInt16())
        cos_list.append(cos_with_year)
    cos_ic = ee.ImageCollection.fromImages(cos_list)
    # Pick minimum cosine (max dissimilarity) via quality mosaic over 1-cos
    scored = cos_ic.map(lambda im: ee.Image(im).addBands(ee.Image(1).subtract(ee.Image(im).select('cos')).rename('score')))
    picked = scored.qualityMosaic('score')
    lowest_sim = picked.select('cos')
    yoc = picked.select('pair_year').rename('alpha_yod')
    mag = ee.Image(1).subtract(lowest_sim).rename('alpha_mag')
    # Duration = count of pairs with dissimilarity above threshold
    inv_sorted = cos_ic.sort('pair_year').map(lambda im: ee.Image(1).subtract(ee.Image(im).select('cos')).rename('inv'))
    mask_ic = inv_sorted.map(lambda im: ee.Image(im).gt(change_threshold).rename('mask').toInt16())
    dur = mask_ic.sum().toInt16().rename('alpha_dur')
    return yoc, mag, dur

def smooth_and_mask_alpha(yoc, mag, dur, mag_threshold=0.15, radius=2, sigma=1.0):
    kernel = ee.Kernel.gaussian(radius=radius, sigma=sigma, units='pixels')
    mag_s = mag.convolve(kernel)
    change_mask = mag_s.gt(mag_threshold)
    yoc_m = yoc.updateMask(change_mask)
    mag_m = mag_s.updateMask(change_mask)
    dur_m = dur.updateMask(change_mask)
    return mag_s, change_mask, yoc_m, mag_m, dur_m

In [9]:
# Interactive map with draw + compute UI
Map = geemap.Map(center=[37.5, -98], zoom=4)
Map.add_basemap('Esri.WorldImagery')
Map.add_layer_control()

draw = DrawControl(
    polyline={},
    circlemarker={},
    rectangle={"shapeOptions": {"color": "#00ffff", "opacity": 1.0, "weight": 2, "fillColor": "#00ffff", "fillOpacity": 0.0}},
    polygon={"shapeOptions": {"color": "#00ffff", "opacity": 1.0, "weight": 2, "fillColor": "#00ffff", "fillOpacity": 0.0}},
    circle={}
 )
Map.add_control(draw)

# AOI state
aoi_geom = {'ee': None}

status = widgets.HTML(value="<b>Step 1:</b> Draw a polygon/rectangle AOI on the map.")
btn_clear = widgets.Button(description='Clear AOI', button_style='warning')
btn_run = widgets.Button(description='Compute change', button_style='primary')

# Parameters
mag_thresh = widgets.FloatSlider(description='AE MAG thr', value=0.15, min=0.05, max=0.5, step=0.01, readout_format='.2f')
smooth_radius = widgets.IntSlider(description='Smooth radius', value=2, min=0, max=5)
smooth_sigma = widgets.FloatSlider(description='Smooth sigma', value=1.0, min=0.25, max=3, step=0.25)
opacity = widgets.FloatSlider(description='Layer opacity', value=0.85, min=0.2, max=1.0, step=0.05)

ui = VBox([status, HBox([btn_clear, btn_run]), HBox([mag_thresh, smooth_radius, smooth_sigma, opacity])])

# Capture draw events
def _to_ee_geometry(feature):
    geom_type = feature['geometry']['type']
    coords = feature['geometry']['coordinates']
    if geom_type == 'Polygon':
        return ee.Geometry.Polygon(coords)
    if geom_type == 'Rectangle':  # ipyleaflet sends Polygon for rect; kept for future
        return ee.Geometry.Polygon(coords)
    if geom_type == 'LineString':
        return ee.Geometry.LineString(coords)
    if geom_type == 'Point':
        return ee.Geometry.Point(coords)
    # default try polygon
    return ee.Geometry.Polygon(coords)

def _on_draw(target, action=None, geo_json=None):
    if action in ('created', 'edited') and geo_json:
        try:
            geom = _to_ee_geometry(geo_json)
            aoi_geom['ee'] = geom
            status.value = "<b>AOI set.</b> Click 'Compute change' to run AlphaEarth method."
        except Exception as e:
            status.value = f"Error parsing geometry: {e}"
    elif action == 'deleted':
        aoi_geom['ee'] = None
        status.value = "AOI cleared. Draw again."

draw.on_draw(_on_draw)

def _clear_layers(prefixes=('AE: ',)):
    # Remove previous layers matching prefixes to keep map tidy
    to_remove = []
    for lyr in list(Map.layers):
        name = getattr(lyr, 'name', '') or ''
        if any(name.startswith(p) for p in prefixes):
            to_remove.append(lyr)
    for lyr in to_remove:
        try:
            Map.remove_layer(lyr)
        except Exception:
            pass

def _run_compute(_):
    if aoi_geom['ee'] is None:
        status.value = "<span style='color:red'>Please draw an AOI first.</span>"
        return
    status.value = "Running computation on server (AlphaEarth)… please wait…"
    geom = aoi_geom['ee']

    # Remove previously added AE layers for clean re-run
    _clear_layers()

    # Compute raw AlphaEarth layers
    yoc, mag, dur = compute_alpha_layers(geom, change_threshold=mag_thresh.value)
    # Add raw layers (clear names, consistent order)
    Map.addLayer(yoc, vis_yod, 'AE: Raw - YOC', False, opacity.value)
    Map.addLayer(mag, vis_mag_a, 'AE: Raw - MAG', False, opacity.value)
    Map.addLayer(dur, vis_dur_a, 'AE: Raw - DUR', False, opacity.value)

    # Smooth + mask
    mag_s, chg_mask, yoc_m, mag_m, dur_m = smooth_and_mask_alpha(
        yoc, mag, dur, mag_threshold=mag_thresh.value, radius=smooth_radius.value, sigma=smooth_sigma.value
    )
    # Add masked/smoothed layers (decision-ready)
    Map.addLayer(yoc_m, vis_yod, 'AE: Masked - YOC', False, opacity.value)
    Map.addLayer(mag_m, vis_mag_a, 'AE: Masked - MAG', True, opacity.value)
    Map.addLayer(dur_m, vis_dur_a, 'AE: Masked - DUR', False, opacity.value)
    Map.addLayer(mag_s, vis_mag_a, 'AE: Smoothed - MAG', False, opacity.value)
    Map.addLayer(chg_mask.selfMask(), {'min':0, 'max':1, 'palette':['#00ff00']}, 'AE: Change Mask', False, 0.8)

    # Outline AOI for reference (no fill)
    Map.addLayer(ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(geom)]), 1, 2),
                  {'palette': ['#00ffff']}, 'AE: AOI Outline', True)

    status.value = "Done. Toggle raw vs masked layers in the control to compare."

btn_run.on_click(_run_compute)
btn_clear.on_click(lambda _:(draw.clear(), _clear_layers(), aoi_geom.update({'ee': None}), setattr(status, 'value', 'AOI cleared. Draw again.')))

display(ui)
Map

Map(center=[37.5, -98], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', t…